# Extract one day of Locomizer signals from Allas

Reads one month of raw signals for one country from Allas (about 112 GiB for Germany) and keeps only the rows of **one calendar day**,
written as parquet files into scratch. The exploration notebook (`explore_munich_weekday.ipynb`) then reads that extract.

**How it works**, batch by batch:
1. copy a batch of parquet files from Allas to the node-local disk (`$TMPDIR`)
2. filter the rows of the chosen day with DuckDB and keep the useful columns
3. write one parquet file per batch into `/scratch/project_2022768/<user>/locomizer/<COUNTRY>_signals_<DATE>/`
4. delete the batch from the local disk and continue

Batches that are already written are skipped, so the notebook can simply be run again if it was interrupted.

**How to run.** This needs a compute node with ~16 cores and ~48 GB for about 20–30 minutes (roughly 10–36 billing units).
Either open it in Jupyter at <https://www.roihu.csc.fi> with those resources and run all cells, or let it run as a batch job
(the "Submit as a batch job" section at the end shows the command).

In [ ]:
# ---- parameters -------------------------------------------------------------
COUNTRY     = "DEU"        # DEU, GBR or FRA (see COUNTRIES below)
MONTH       = "2024-05"    # YYYY-MM
DAY         = 15           # day of the month to keep
BATCH_GIB   = 8            # how much input to download per batch (must fit in $TMPDIR)
MAX_BATCHES = 0            # 0 = whole month; e.g. 2 = smoke test on the first 2 batches only

In [ ]:
import os, shutil, subprocess, time
from pathlib import Path
import duckdb

# Allas access goes through the rclone remotes created by `allas-conf -m S3 <project>` (remote name s3allas-<project>).
# Nothing here touches credentials.
COUNTRIES = {
    "DEU": {"project": "project_2014316", "bucket": "profile_signals_DEU", "prefix": "profile_signals_DEU"},
    "GBR": {"project": "project_2014314", "bucket": "profile_signals_GBR", "prefix": "profile_signals_GBR"},
    "FRA": {"project": "project_2014315", "bucket": "profile_signals_FRA", "prefix": "profile_signals_FRA"},
}
SCRATCH = Path(f"/scratch/project_2022768/{os.environ['USER']}/locomizer")
RCLONE  = shutil.which("rclone") or "/appl/soft/manual/general/x86_64/allas-tools/bin/rclone"   # from `module load allas`

def remote_path(country, dataset="signals_md5", month=None):
    c = COUNTRIES[country]
    p = f"s3allas-{c['project']}:{c['bucket']}/{c['prefix']}/{dataset}"
    return f"{p}/{month}" if month else p

def list_parts(country, month, dataset="signals_md5"):
    """[(filename, bytes), ...] for one month of a dataset (read-only listing)."""
    res = subprocess.run([RCLONE, "ls", remote_path(country, dataset, month)], capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(res.stderr)
    return sorted((name.strip(), int(size)) for size, name in (l.split(maxsplit=1) for l in res.stdout.splitlines()))

DATE    = f"{MONTH}-{DAY:02d}"
threads = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count()))
tmp     = Path(os.environ.get("TMPDIR", "/tmp")) / "loco_batch"
out_dir = SCRATCH / f"{COUNTRY}_signals_{DATE}"
out_dir.mkdir(parents=True, exist_ok=True)
print(f"output -> {out_dir}\nthreads = {threads}, local disk = {tmp}")

## 1. What is in the bucket

In [3]:
parts = list_parts(COUNTRY, MONTH)
total = sum(s for _, s in parts)
print(f"{COUNTRY} signals_md5/{MONTH}: {len(parts)} parquet parts, {total/2**30:.1f} GiB")

# split into batches that fit comfortably on the local disk
batches, cur, cur_size = [], [], 0
for name, size in parts:
    if cur and cur_size + size > BATCH_GIB * 2**30:
        batches.append(cur); cur, cur_size = [], 0
    cur.append(name); cur_size += size
if cur:
    batches.append(cur)
if MAX_BATCHES:
    batches = batches[:MAX_BATCHES]
    print(f"SMOKE TEST: only {len(batches)} batch(es)")
print(f"{len(batches)} batches of up to {BATCH_GIB} GiB")

DEU signals_md5/2024-05: 465 parquet parts, 112.6 GiB
15 batches of up to 8 GiB


## 2. Download, filter, write

In [4]:
con = duckdb.connect()
con.execute(f"SET threads = {threads}; SET preserve_insertion_order = false;")
t0 = time.time()
for i, batch in enumerate(batches):
    out_file = out_dir / f"part-{i:04d}.parquet"
    if out_file.exists():
        print(f"batch {i+1}/{len(batches)}: exists, skipping", flush=True)
        continue
    shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    (tmp / "files.txt").write_text("\n".join(batch) + "\n")
    subprocess.run([RCLONE, "copy", remote_path(COUNTRY, "signals_md5", MONTH), str(tmp / "in"),
                    "--files-from", str(tmp / "files.txt"), "--transfers", "8", "--checkers", "8", "--quiet"], check=True)
    partial = str(out_file) + ".partial"
    con.execute(f"""
        COPY (
            SELECT Userid AS userid, lat, lon, timestamp, datetime, dow, hour, minute,
                   gid AS h3_9, "10lvl_gid" AS h3_10, track_type
            FROM read_parquet('{tmp}/in/*.parquet')
            WHERE day = {DAY} AND month = {int(MONTH.split('-')[1])}
        ) TO '{partial}' (FORMAT parquet, COMPRESSION zstd);
    """)
    os.replace(partial, out_file)
    n = con.execute(f"SELECT count(*) FROM read_parquet('{out_file}')").fetchone()[0]
    print(f"batch {i+1}/{len(batches)}: {len(batch)} files -> {n:,} rows  ({time.time()-t0:.0f} s elapsed)", flush=True)
shutil.rmtree(tmp, ignore_errors=True)

batch 1/15: 33 files -> 3,011,038 rows  (6 s elapsed)


batch 2/15: 33 files -> 3,009,171 rows  (13 s elapsed)


batch 3/15: 33 files -> 3,009,077 rows  (19 s elapsed)


batch 4/15: 33 files -> 3,009,574 rows  (25 s elapsed)


batch 5/15: 33 files -> 3,005,528 rows  (32 s elapsed)


batch 6/15: 33 files -> 3,010,874 rows  (40 s elapsed)


... (5 outputs left out)

batch 12/15: 33 files -> 3,011,163 rows  (82 s elapsed)


batch 13/15: 33 files -> 3,009,220 rows  (89 s elapsed)


batch 14/15: 33 files -> 3,010,064 rows  (96 s elapsed)


batch 15/15: 3 files -> 273,700 rows  (99 s elapsed)


## 3. Result

In [5]:
n, u = con.execute(f"SELECT count(*), count(DISTINCT userid) FROM read_parquet('{out_dir}/*.parquet')").fetchone()
size = sum(f.stat().st_size for f in out_dir.glob("*.parquet"))
print(f"DONE {DATE}: {n:,} signals, {u:,} distinct users, {size/2**30:.2f} GiB in {len(list(out_dir.glob('*.parquet')))} files -> {out_dir}")

DONE 2024-05-15: 42,414,993 signals, 1,134,577 distinct users, 1.69 GiB in 15 files -> /scratch/project_2022768/marinflo/locomizer/DEU_signals_2024-05-15


## Submit as a batch job

To run this notebook as a batch job instead of interactively, from a terminal on `roihu-cpu.csc.fi` (one line, no script file needed):

```bash
sbatch --account=project_2022768 --partition=small --time=02:00:00 --cpus-per-task=16 --mem=48G --job-name=extract_day \
  --output=/scratch/project_2022768/%u/locomizer/logs/extract_%j.out \
  --wrap="module load allas python-geo; cd ~/locomizer_signals/pipeline; jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 extract_day.ipynb"
```

The executed notebook (with its printed progress) is saved in place when the job ends.